# Preprocesamiento - Rendimiento Académico

# 1.Importación librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2.Importación fuente de datos

In [2]:
df_mat = pd.read_csv('../data/raw/student-mat.csv', sep=';')
df_por = pd.read_csv('../data/raw/student-por.csv', sep=';')

print(df_mat.shape, df_por.shape)

(395, 33) (649, 33)


## 2.1 Unión de las dos materias

In [3]:
df_mat['materia'] = 'matematicas'
df_por['materia'] = 'portugues'

df = pd.concat([df_mat, df_por], ignore_index=True)

print(df.shape)
df.head()

(1044, 34)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,materia
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,matematicas
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,matematicas
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,matematicas
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,matematicas
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,matematicas


# 3.Análisis de Datos (EDA) + Preprocesado

## 3.1 Limpieza inicial

In [4]:
df.info()
print()
print("Nulos por columna:")
print(df.isnull().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1044 entries, 0 to 1043
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      1044 non-null   str  
 1   sex         1044 non-null   str  
 2   age         1044 non-null   int64
 3   address     1044 non-null   str  
 4   famsize     1044 non-null   str  
 5   Pstatus     1044 non-null   str  
 6   Medu        1044 non-null   int64
 7   Fedu        1044 non-null   int64
 8   Mjob        1044 non-null   str  
 9   Fjob        1044 non-null   str  
 10  reason      1044 non-null   str  
 11  guardian    1044 non-null   str  
 12  traveltime  1044 non-null   int64
 13  studytime   1044 non-null   int64
 14  failures    1044 non-null   int64
 15  schoolsup   1044 non-null   str  
 16  famsup      1044 non-null   str  
 17  paid        1044 non-null   str  
 18  activities  1044 non-null   str  
 19  nursery     1044 non-null   str  
 20  higher      1044 non-null   str  
 21  in

In [5]:
print("Filas duplicadas:", df.duplicated().sum())
df.describe().transpose()

Filas duplicadas: 0


,count,mean,std,min,25%,50%,75%,max
age,1044.0,16.726054,1.239975,15.0,16.0,17.0,18.0,22.0
Medu,1044.0,2.603448,1.124907,0.0,2.0,3.0,4.0,4.0
Fedu,1044.0,2.387931,1.099938,0.0,1.0,2.0,3.0,4.0
traveltime,1044.0,1.522989,0.731727,1.0,1.0,1.0,2.0,4.0
studytime,1044.0,1.970307,0.834353,1.0,1.0,2.0,2.0,4.0
failures,1044.0,0.264368,0.656142,0.0,0.0,0.0,0.0,3.0
famrel,1044.0,3.935824,0.933401,1.0,4.0,4.0,5.0,5.0
freetime,1044.0,3.201149,1.031507,1.0,3.0,3.0,4.0,5.0
goout,1044.0,3.156130,1.152575,1.0,2.0,3.0,4.0,5.0
Dalc,1044.0,1.494253,0.911714,1.0,1.0,1.0,2.0,5.0


In [6]:
print("Estudiantes con G3 = 0:", (df['G3'] == 0).sum())
print("Estudiantes con absences >= 30:", (df['absences'] >= 30).sum())

df[df['G3'] == 0][['materia', 'absences', 'G1', 'G2', 'G3']].head(10)

Estudiantes con G3 = 0: 53
Estudiantes con absences >= 30: 8


,materia,absences,G1,G2,G3
128,matematicas,0,7,4,0
130,matematicas,0,12,0,0
131,matematicas,0,8,0,0
134,matematicas,0,9,0,0
135,matematicas,0,11,0,0
136,matematicas,0,10,0,0
137,matematicas,0,4,0,0
140,matematicas,0,7,9,0
144,matematicas,0,5,0,0
146,matematicas,0,6,7,0


In [7]:
ceros = df[df['G3'] == 0]

print("Por materia:")
print(ceros['materia'].value_counts())
print()
print("Con absences = 0:", (ceros['absences'] == 0).sum())
print("Con G2 = 0:", (ceros['G2'] == 0).sum())

Por materia:
materia
matematicas    38
portugues      15
Name: count, dtype: int64

Con absences = 0: 53
Con G2 = 0: 20


**Hallazgos de la limpieza inicial**

- No hay valores nulos ni filas duplicadas en los 1044 registros.
- Los rangos de las variables son coherentes (notas entre 0 y 20).
- Hay 53 estudiantes con G3 = 0 (38 de matemáticas y 15 de portugués). Todos tienen 0 faltas y 20 tienen también G2 = 0.
- Hay 8 estudiantes con 30 o más faltas (el máximo es 75).
- Decisión: por ahora no se eliminan estos casos, porque representan estudiantes en riesgo alto que el modelo debe aprender a detectar.

## 3.3 Limpieza final

### 3.3.1 Creación de variables derivadas (Feature engineering)

### 3.3.2 Creación de la variable objetivo del Modelo 2 (Nivel de riesgo)

### 3.3.3 Transformación variables categóricas

# 4.Train Test Split

# 5.Escalado

# 6.Guardar datos procesados